# 🎬 EDS Colab Processing Server
## Emotion Data Studio — GPU Processing on Google Colab

**Chạy trên Colab Pro (GPU T4/V100/A100)**: Pipeline AI tốc độ cao, truy cập dashboard từ trình duyệt bất kỳ đâu qua ngrok.

**Sau khi chạy xong**: Dashboard sẽ có URL ngrok ở cell cuối cùng.

In [ ]:
# ============================================================
# 0. CÀI ĐẶT MÔI TRƯỜNG
# ============================================================

# Upgrade pip
!pip install -q --upgrade pip

# Install core dependencies
!pip install -q \
  fastapi uvicorn \
  sqlalchemy pydantic pydantic-settings \
  transformers datasets accelerate \
  torch torchaudio \
  librosa opencv-python \
  yt-dlp gdown \
  google-cloud-storage google-cloud-sql-connector \
  psycopg2-binary pg8000 \
  pyngrok httpx \
  google-cloud-aiplatform>=2.0.0

# Verify GPU
import torch
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"✅ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — processing will be slow. Use a GPU runtime.")

In [ ]:
# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
EDS_ROOT = '/content/drive/MyDrive/EDS'
os.makedirs(EDS_ROOT, exist_ok=True)
os.makedirs(f'{EDS_ROOT}/videos', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/clips', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/audio', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/features', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/exports', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/data', exist_ok=True)

print(f"✅ Drive mounted: {EDS_ROOT}")

In [ ]:
# ============================================================
# 2. CLONE / PULL EDS REPO
# ============================================================

# Nếu chưa clone: thay YOUR_REPO_URL bằng URL repo thật của bạn
REPO_URL = "https://github.com/YOUR_USERNAME/BCDA.git"  # <-- THAY ĐỔI
REPO_DIR = '/content/BCDA'

import subprocess, os

if os.path.exists(REPO_DIR):
    print("📁 Repo đã tồn tại — pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=False)
else:
    print("📥 Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Add repo to path
import sys
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/tools/emotion-data-studio')

print(f"✅ Repo ready: {REPO_DIR}")

In [ ]:
# ============================================================
# 4. CẤU HÌNH GOOGLE CLOUD & GEMINI AUTO-LABELER
# ============================================================
#
# 1. Tạo service account tại: https://console.cloud.google.com/iam-admin/serviceaccounts
#    Roles: Storage Admin + Vertex AI User
# 2. Tải JSON key → upload lên Drive: /content/drive/MyDrive/EDS/credentials/
# 3. Tạo GCS bucket: gs://your-bucket-name
# 4. Điền GCP_PROJECT_ID, GCS_BUCKET_NAME bên dưới

import os

SERVICE_ACCOUNT_KEY = "/content/drive/MyDrive/EDS/credentials/service-account.json"
if os.path.exists(SERVICE_ACCOUNT_KEY):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SERVICE_ACCOUNT_KEY
    print(f"✅ Service account: OK")
else:
    print("⚠️  Service account key chưa có!")

os.environ["GCP_PROJECT_ID"] = "your-gcp-project-id"
os.environ["GCP_LOCATION"] = "us-central1"
os.environ["GCS_BUCKET_NAME"] = "your-bucket-name-emotion-data"

os.environ["GEMINI_MODEL"] = "gemini-2.5-flash"
os.environ["GEMINI_TEMPERATURE"] = "0.2"
os.environ["GEMINI_MAX_TOKENS"] = "8192"
os.environ["GEMINI_INTENSITY_THRESHOLD"] = "0.6"

os.environ["EDS_DATA_DIR"] = f"{EDS_ROOT}/data"
os.environ["EDS_DOWNLOAD_MODE"] = "balanced"
os.environ["EDS_DOWNLOAD_MAX_HEIGHT"] = "720"

!pip install -q google-cloud-aiplatform>=2.0.0

try:
    from backend.services.gemini_auto_labeler import GeminiAutoLabeler
    labeler = GeminiAutoLabeler()
    print(f"🤖 Gemini: {labeler.status()}")
except Exception as e:
    print(f"⚠️  Gemini config error: {e}")

print("✅ Google Cloud & Gemini configured")

In [ ]:
# ============================================================
# 4. PREWARM MODELS (GPU)
# ============================================================

import sys, logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('EDS')

sys.path.insert(0, f'{REPO_DIR}/tools/emotion-data-studio')

# Init database
from backend.database.local_db import init_database, get_session
init_database()
print("✅ Database initialized")

# Prewarm core models (skip text/audio emotion for now — loaded lazily)
from backend.ai_models.model_manager import model_manager

logger.info("Prewarming Whisper (medium)...")
try:
    model_manager.prewarm_models(['whisper', 'deepface', 'mtcnn'])
    logger.info("✅ Core models loaded")
except Exception as e:
    logger.warning(f"⚠️ Some models failed: {e}")

# Load text/audio emotion models (optional)
logger.info("Loading text/audio emotion models...")
try:
    model_manager.prewarm_models(['text_emotion', 'audio_emotion'])
    logger.info("✅ Emotion models loaded")
except Exception as e:
    logger.warning(f"⚠️ Emotion models failed: {e}")

print(model_manager.status())

In [ ]:
# ============================================================
# 5. KHỞI ĐỘNG WEB DASHBOARD
# ============================================================

import subprocess, threading, time

# Start ngrok in background
# 1. Get ngrok auth token: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "YOUR_NGROK_TOKEN"  # <-- THAY ĐỔI với token thật của bạn

if NGROK_TOKEN == "YOUR_NGROK_TOKEN":
    print("⚠️  Chưa đặt NGROK_TOKEN! Dashboard sẽ chạy ở port 8765 (không public).")
    print("   Lấy token tại: https://dashboard.ngrok.com/get-started/your-authtoken")
    public_url = None
else:
    !pip install -q pyngrok
    from pyngrok import ngrok
    
    # Kill any existing tunnels
    ngrok.kill()
    
    # Set token
    ngrok.set_auth_token(NGROK_TOKEN)
    
    # Create tunnel to port 8765
    tunnel = ngrok.connect(addr="8765", proto="http", bind_tls=True)
    public_url = tunnel.public_url
    print(f"🌐 Dashboard: {public_url}")

# Start FastAPI server
import os
os.chdir(f'{REPO_DIR}/tools/emotion-data-studio')

def run_server():
    import uvicorn
    uvicorn.run(
        "web.main:app",
        host="0.0.0.0",
        port=8765,
        log_level="info",
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(3)
print("✅ Web server started on port 8765")

In [ ]:
# ============================================================
# 6. START PIPELINE WORKER (BACKGROUND)
# ============================================================

import threading, time, logging

logger = logging.getLogger('Pipeline')

def run_pipeline():
    from backend.database.local_db import get_session
    from backend.database.models import ProcessQueue
    from backend.services.pipeline_orchestrator import PipelineOrchestrator
    from datetime import datetime
    
    while True:
        try:
            session = get_session()
            try:
                item = (
                    session.query(ProcessQueue)
                    .filter(ProcessQueue.status == "queued")
                    .order_by(ProcessQueue.priority.desc(), ProcessQueue.created_at.asc())
                    .first()
                )
                if not item:
                    time.sleep(5)
                    continue

                item.status = 'running'
                item.started_at = datetime.utcnow()
                session.commit()

                logger.info(f"Processing video {item.video_id}...")
                orchestrator = PipelineOrchestrator()
                orchestrator.process_video(video_id=item.video_id)

                item.status = 'done'
                item.completed_at = datetime.utcnow()
                session.commit()
                logger.info(f"Video {item.video_id} done!")

            except Exception as e:
                logger.error(f"Pipeline error: {e}")
                if item:
                    item.status = 'error'
                    item.error_message = str(e)
                    session.commit()
            finally:
                session.close()

        except Exception as e:
            logger.error(f"Outer loop error: {e}")
            time.sleep(10)

pipeline_thread = threading.Thread(target=run_pipeline, daemon=True)
pipeline_thread.start()

print("✅ Pipeline worker started in background")

In [ ]:
# ============================================================
# 7. DASHBOARD ACCESS
# ============================================================

print("=" * 60)
print("🎉 EDS COLAB DASHBOARD SẴN SÀNG!")
print("=" * 60)

if 'public_url' in dir() and public_url:
    print(f"\n🌐 Dashboard: {public_url}")
    print(f"   └── /               → Dashboard")
    print(f"   └── /review         → Review clips")
    print(f"   └── /harvest        → Thu hoạch video")
    print(f"   └── /gemini         → Gemini Auto-Label ⭐")
    print(f"   └── /settings       → Cài đặt")
    print(f"   └── /export         → Xuất dataset")
    print(f"   └── /api/gemini/status → Gemini status (JSON)")
    print(f"   └── /api/worker/status → Colab Worker status (JSON)")
    print(f"   └── /api/worker/claim  → Worker claim endpoint (internal)")
else:
    print("\n⚠️  No ngrok URL — Dashboard chỉ truy cập được từ Colab.")
    print("   Truy cập: http://localhost:8765")

print("\n📝 HƯỚNG DẪN GEMINI AUTO-LABEL:")
print("   1. Tab Gemini Auto-Label: paste video path, nhấn Phân Tích")
print("   2. Xem các đoạn cảm xúc mạnh + nhãn emotion")
print("   3. Nhấn 'Áp Dụng' để tạo clips trong DB")
print(f"\n   💰 Chi phí: ~$0.01/clip | Ngân sách: $500/tháng")
print("\n🔗 Ngrok: https://dashboard.ngrok.com/get-started/your-authtoken")
print("🔗 GCP: https://console.cloud.google.com/iam-admin/serviceaccounts")
print("=" * 60)

In [ ]:
# ============================================================
# 8. QUẢN LÝ COLAB SESSION (tránh disconnect)
# ============================================================

# Chạy cell này để giữ Colab sống + auto-reconnect
# Nhấn Ctrl+Shift+i → Console → paste:
#
# function KeepClicking(){
#   document.querySelector("#connect > div.toolbar-middle > colab-connect-button").click();
#   document.querySelector("colab-output-monitor").shadowRoot.querySelector("#heartbeat").click();
# }
# setInterval(KeepClicking, 60000);

# Hoặc dùng Colab Pro với High-RAM runtime để tránh OOM.
print("💡 Tip: Dùng 'Connect → Connect to hosted runtime' để giữ session.")
print("   Nếu bị disconnect, chạy lại từ cell 7.")

# ============================================================
# 9. COLAB GPU WORKER — KẾT NỐI VỚI LOCAL BACKEND
# ============================================================
#
# ⚡ Worker nhận việc từ local backend qua ngrok tunnel.
#
# Cách hoạt động:
#   1. Chạy local backend: python app.py (port 8765)
#   2. Mở ngrok: ngrok http 8765
#   3. Paste NGROK_TOKEN vào biến bên dưới, chạy cell này
#   4. Colab tự động claim + xử lý jobs từ local backend

import os, sys, logging
REPO_DIR = "/content/BCDA"
sys.path.insert(0, f"{REPO_DIR}/tools/emotion-data-studio")
sys.path.insert(0, REPO_DIR)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("ColabWorker")

import torch
if torch.cuda.is_available():
    log.info(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    log.warning("⚠️ No GPU!")

NGROK_TOKEN = "YOUR_NGROK_TOKEN"  # <-- Đổi token tại đây
LOCAL_BACKEND_PORT = 8765
EDS_DATA_DIR = "/content/drive/MyDrive/EDS/data"
os.makedirs(EDS_DATA_DIR, exist_ok=True)
os.environ["EDS_DATA_DIR"] = EDS_DATA_DIR

if NGROK_TOKEN == "YOUR_NGROK_TOKEN":
    log.warning("⚠️ Chưa đặt NGROK_TOKEN — worker không kết nối được.")
    log.warning("   Token: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    import subprocess
    p = subprocess.Popen(
        ["python", f"{REPO_DIR}/tools/emotion-data-studio/colab/colab_worker.py",
         "--ngrok-token", NGROK_TOKEN,
         "--backend-port", str(LOCAL_BACKEND_PORT),
         "--data-dir", EDS_DATA_DIR,
         "--repo-dir", REPO_DIR],
    )
    log.info(f"Worker started (pid={p.pid})")